# Module 4.5 — Introspection & Reflection

### Python Mastery · Track 4: Functional Programming & Metaprogramming

Introspection is the ability of a program to examine its own objects at runtime: what type something is, what attributes and methods it has, and how a function is meant to be called. Reflection extends this to acting on that information, such as reading or setting attributes by name. This module covers the built-in tools, the `inspect` module, and annotations.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Point the tools at your own objects to see what they reveal.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Examine objects with `type`, `id`, `dir`, and `vars`.
2. Read and set attributes by name with `getattr`, `setattr`, and `hasattr`.
3. Use the `inspect` module to read signatures and source.
4. Distinguish functions, methods, and classes programmatically.
5. Read function and class annotations.

**Prerequisites:** Tracks 1 to 3, and Modules 4.1 to 4.4.

---

## Part 1 · Basic Inspection: `type`, `dir`, `vars`

A handful of built-ins reveal an object's nature. `type(obj)` gives its class; `dir(obj)` lists its attributes and methods (including inherited ones); and `vars(obj)` returns the object's own writable attributes as a dictionary (the same as `obj.__dict__`).

In [ ]:
class Robot:
    category = "machine"          # class attribute
    def __init__(self, name):
        self.name = name          # instance attribute
    def beep(self):
        return "beep"

r = Robot("R2")

print("type:", type(r).__name__)

# dir lists everything accessible, including inherited and dunder members.
public = [name for name in dir(r) if not name.startswith("_")]
print("public members:", public)

# vars shows just this instance's own attributes.
print("instance attributes:", vars(r))

## Part 2 · Reflection: `getattr`, `setattr`, `hasattr`

Reflection lets you work with attributes whose names are decided at runtime, held in variables or strings. `getattr(obj, "x")` reads an attribute by name (with an optional default), `setattr(obj, "x", value)` sets one, and `hasattr(obj, "x")` checks for existence. This enables generic code such as dispatching to a method chosen from input.

In [ ]:
class Robot:
    def __init__(self, name):
        self.name = name
    def beep(self): return "beep"
    def move(self): return "moving"

r = Robot("R2")

# Access an attribute by a name held in a variable.
attr = "name"
print("getattr:", getattr(r, attr))
print("getattr with default:", getattr(r, "color", "unknown"))

# Set an attribute by name.
setattr(r, "battery", 100)
print("after setattr:", r.battery)

# Check before accessing.
print("has beep:", hasattr(r, "beep"))
print("has fly:", hasattr(r, "fly"))

# Dispatch: choose a method by name, then call it.
for command in ["beep", "move"]:
    method = getattr(r, command)
    print(f"{command} -> {method()}")

## Part 3 · The `inspect` Module: Signatures

The `inspect` module provides richer introspection than the built-ins. `inspect.signature(func)` returns a structured description of how a function should be called: its parameters, their kinds, and their defaults. This is invaluable for building tools that adapt to whatever function they are given.

In [ ]:
import inspect

def configure(host, port=8080, *, debug=False, **extra):
    pass

sig = inspect.signature(configure)
print("signature:", sig)

# Examine each parameter individually.
for name, param in sig.parameters.items():
    default = param.default if param.default is not inspect.Parameter.empty else "(required)"
    print(f"  {name}: kind={param.kind.name}, default={default}")

## Part 4 · The `inspect` Module: Classifying and Reading Source

`inspect` can also tell you what kind of object you have, with predicates such as `isfunction`, `ismethod`, and `isclass`, and it can retrieve an object's source code with `getsource`. These power documentation generators, debuggers, and test tools.

In [ ]:
import inspect

def standalone():
    """A module-level function."""
    return 1

class Widget:
    def method(self):
        return 2

w = Widget()

print("standalone is a function:", inspect.isfunction(standalone))
print("w.method is a method:", inspect.ismethod(w.method))
print("Widget is a class:", inspect.isclass(Widget))

# Retrieve the source code of a function as text.
print("--- source of standalone ---")
print(inspect.getsource(standalone))

## Part 5 · Annotations

Type annotations attached to functions and classes are stored and can be read at runtime. A function's annotations live in `__annotations__`; for safe, forward-reference-aware access, `inspect.get_annotations` is preferred. Annotations do not enforce types by themselves, but tools and your own code can read them to validate, document, or generate behaviour.

In [ ]:
import inspect

def transfer(amount: float, to: str, urgent: bool = False) -> str:
    return f"sent {amount} to {to}"

# Raw annotations dictionary.
print("raw annotations:", transfer.__annotations__)

# The recommended accessor (handles forward references cleanly).
print("via inspect:", inspect.get_annotations(transfer))

# Class-level annotations are stored too.
class Account:
    owner: str
    balance: float = 0.0

print("class annotations:", inspect.get_annotations(Account))

# A small use: read annotations to coerce inputs to their declared types.
def coerce_args(func, **kwargs):
    hints = inspect.get_annotations(func)
    return {key: hints[key](value) if key in hints else value
            for key, value in kwargs.items()}

print("coerced:", coerce_args(transfer, amount="50", to="Asha"))   # amount becomes a float

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Listing an object's methods (Easy)

In [ ]:
class Calculator:
    def add(self, a, b): return a + b
    def subtract(self, a, b): return a - b

c = Calculator()
methods = [name for name in dir(c) if not name.startswith("_")]
print("methods:", methods)

### Example 2 — Reading attributes by name (Easy)

In [ ]:
class Config:
    host = "localhost"
    port = 8080

for setting in ["host", "port", "missing"]:
    print(setting, "->", getattr(Config, setting, "not set"))

### Example 3 — Command dispatch by reflection (Medium)

In [ ]:
class Light:
    def __init__(self):
        self.on = False
    def turn_on(self): self.on = True; return "light on"
    def turn_off(self): self.on = False; return "light off"
    def status(self): return f"on={self.on}"

light = Light()
for command in ["turn_on", "status", "turn_off", "status"]:
    if hasattr(light, command):
        print(command, "->", getattr(light, command)())

### Example 4 — Inspecting a function's parameters (Medium)

In [ ]:
import inspect

def make_user(name, age=18, *, active=True):
    pass

sig = inspect.signature(make_user)
required = [n for n, p in sig.parameters.items()
            if p.default is inspect.Parameter.empty and p.kind != p.VAR_KEYWORD]
optional = [n for n, p in sig.parameters.items()
            if p.default is not inspect.Parameter.empty]
print("required:", required)
print("optional:", optional)

### Example 5 — A generic object describer (Difficult)

In [ ]:
import inspect

def describe(obj):
    """Print a structured summary of any object."""
    print(f"type: {type(obj).__name__}")
    attributes = {name: getattr(obj, name) for name in vars(obj)} if hasattr(obj, "__dict__") else {}
    print(f"data attributes: {attributes}")
    methods = [name for name in dir(obj)
               if callable(getattr(obj, name)) and not name.startswith("_")]
    print(f"methods: {methods}")

class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages
    def summary(self):
        return f"{self.title} ({self.pages}p)"

describe(Book("Dune", 412))

### Example 6 — Validating arguments against annotations (Difficult)

In [ ]:
import inspect

def checked(func):
    """A decorator that verifies arguments match the function's annotations."""
    hints = inspect.get_annotations(func)
    sig = inspect.signature(func)

    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)        # match arguments to parameter names
        for name, value in bound.arguments.items():
            if name in hints and not isinstance(value, hints[name]):
                raise TypeError(f"{name} should be {hints[name].__name__}")
        return func(*args, **kwargs)
    return wrapper

@checked
def register(name: str, age: int) -> str:
    return f"{name} is {age}"

print(register("Asha", 30))
try:
    register("Ben", "old")                       # age is not an int
except TypeError as e:
    print("rejected:", e)

---

## Exercises

**Exercise 1 (Easy).** Given the class below, print a list of its public (non-underscore) attribute and method names using `dir`.

In [ ]:
class Phone:
    brand = "Acme"
    def call(self): pass
    def text(self): pass
# Your solution here


**Exercise 2 (Easy).** Using `getattr`, read the attribute whose name is stored in the variable `field` from the object below, with a default of `"n/a"` if it is missing.

In [ ]:
class Person:
    name = "Asha"
field = "name"
# Your solution here


**Exercise 3 (Medium).** Write a function `call_method(obj, method_name, *args)` that calls the named method on `obj` if it exists (using `hasattr` and `getattr`), or returns `"no such method"` otherwise. Test it on a small class.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Using `inspect.signature`, write a function that returns the number of required positional parameters of a given function (those without defaults). Test it on a function with mixed parameters.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Using `inspect.get_annotations`, write a function `summarise(func)` that returns a string listing each annotated parameter and its declared type name, plus the return type. Test it on an annotated function.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
class Phone:
    brand = "Acme"
    def call(self): pass
    def text(self): pass

print([name for name in dir(Phone) if not name.startswith("_")])

**Solution 2**

In [ ]:
class Person:
    name = "Asha"
field = "name"
print(getattr(Person, field, "n/a"))

**Solution 3**

In [ ]:
def call_method(obj, method_name, *args):
    if hasattr(obj, method_name):
        return getattr(obj, method_name)(*args)
    return "no such method"

class Greeter:
    def hello(self, name): return f"hi {name}"

g = Greeter()
print(call_method(g, "hello", "Asha"))
print(call_method(g, "bye", "Asha"))

**Solution 4**

In [ ]:
import inspect

def required_positional(func):
    sig = inspect.signature(func)
    return sum(1 for p in sig.parameters.values()
               if p.default is inspect.Parameter.empty
               and p.kind in (p.POSITIONAL_OR_KEYWORD, p.POSITIONAL_ONLY))

def example(a, b, c=3, *args):
    pass

print(required_positional(example))   # 2

**Solution 5**

In [ ]:
import inspect

def summarise(func):
    hints = inspect.get_annotations(func)
    return_type = hints.pop("return", None)
    parts = [f"{name}: {typ.__name__}" for name, typ in hints.items()]
    result = ", ".join(parts)
    if return_type:
        result += f" -> {return_type.__name__}"
    return result

def transfer(amount: float, to: str) -> bool:
    return True

print(summarise(transfer))

---

## Summary and Key Points

- `type`, `dir`, and `vars` reveal an object's class, all accessible members, and its own attributes respectively.
- `getattr`, `setattr`, and `hasattr` read, write, and check attributes by name, enabling generic code such as dispatch from a string.
- `inspect.signature` returns a structured view of a function's parameters, kinds, and defaults, ideal for tools that adapt to arbitrary functions.
- `inspect` predicates (`isfunction`, `ismethod`, `isclass`) classify objects, and `inspect.getsource` retrieves source code as text.
- Annotations are stored on functions and classes; read them with `__annotations__` or, preferably, `inspect.get_annotations`, and use them to validate, document, or coerce. Annotations do not enforce types on their own.

### Next module: 4.6 — Code Generation & DSLs

The next module covers generating and analysing code at runtime: `exec`, `eval`, and `compile`, the abstract syntax tree via the `ast` module, and building small internal domain-specific languages.